# Simulación de cuatro rollos en horno tipo campana.

## Imports

In [1]:
from copy import deepcopy
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from heattransfer.simulator import Simulator
from heattransfer.models.furnace import Furnace
from heattransfer.models.material import Material
from heattransfer.models.coil import Coil
from heattransfer.models.protection_gas import ProtectionGas
from heattransfer.analysis.analyzer import Analyzer

## Configuración

In [2]:
# Steel SPCC-PSR
material = Material(
    density=7870.0,
    emissivity=0.08554, # (Yang. et al)
    specific_heat=481.0, # (Ltd, 2023)
    hardness=1133.86, # (Yang. et al)
    roughness=3.2215e-6,
    pressure=0
)

gas = ProtectionGas(
   conductivity=0.1805,
   viscosity=8.9e-6,
   density=0.0819
)

coil = Coil(
    material=material,
    inner_radius=0.605 / 2,
    outer_radius=1.837 / 2,
    strip_thickness=1.48e-3,
    height=0.636*2,
    point_separation=0.050,
    initial_temperature=(273.15 + 55),
    surface_slope=0.1,
    oxide_thickness=10e-6, # (Yang. et al)
    oxide_conductivity=3.0,
    emulsion_correction=0.8
)

# Four identical coils
coils = [coil] + [deepcopy(coil) for _ in range(3)]

# Import sample temperature from file and clean it up
gas_temp = np.genfromtxt("./data/experimental/Thermocouples/1#.csv", delimiter=",")
gas_temp = gas_temp[~np.isnan(gas_temp).any(axis=1)]


furnace = Furnace(
    temperature=gas_temp,
    height=5.59, # (Zhang et. al)
    radius=1.03, # (Zhang et. al)
    coil_sep_height=0.10, # (Zhang et. al)
    coils=coils,
    gas=gas,
    gas_flow=19.4, # (Zhang et. al)
    vortex_enhancement=1.0, # (Zhang et. al)
    radiation_enhancement=1.1,
)

simulator = Simulator(
    furnace = furnace, 
    time_step = 25, 
    simulation_time = furnace.duration
)

## Simulate
Activar simulator.run().

In [3]:
results = simulator.run(parallel=True, max_workers=4)

## Análisis

### Settings

In [4]:
from pathlib import Path
from plotly.colors import qualitative

# coil_num is 1-based; r is distance from the inner surface (m),
# z is height from the bottom of the selected coil (m).
# Points other than 9 and 11 are placeholders to replace with your locations.
comparison_points = {
    2:  {"coil_num": 4, "r": 0.015, "z": 0.636*2 - 0.001},
    3:  {"coil_num": 4, "r": coil.outer_radius - coil.inner_radius - 0.003, "z": 0.636*2 - 0.001},
    5:  {"coil_num": 4, "r": coil.outer_radius - coil.inner_radius - 0.003, "z": 0.636},
    6:  {"coil_num": 3, "r": coil.outer_radius - coil.inner_radius - 0.0045, "z": 0.636*2 - 0.001},
    8:  {"coil_num": 2, "r": coil.outer_radius - coil.inner_radius - 0.0045, "z": 0.636*2 - 0.001},
    9:  {"coil_num": 2, "r": 0.309, "z": 0.636},
    11: {"coil_num": 1, "r": 0.312, "z": 0.636},
}

experimental_directory = Path("data/experimental/Thermocouples")
experimental_files = sorted(
    (path for path in experimental_directory.glob("*#.csv")
     if path.stem != "1#"),
    key=lambda path: int(path.stem.removesuffix("#")),
)
if not experimental_files:
    raise ValueError(f"No thermocouple CSV files found in {experimental_directory}")

simulation_hours = np.asarray(simulator.times) / 3600.0
comparisons = []
for index, path in enumerate(experimental_files):
    number = int(path.stem.removesuffix("#"))
    if number not in comparison_points:
        raise ValueError(f"Add a comparison_points entry for Termopar {number}")
    point = comparison_points[number]
    coil_index = point["coil_num"] - 1
    if not 0 <= coil_index < len(furnace.coils):
        raise ValueError(f"Invalid coil_num for Termopar {number}")
    current_coil = furnace.coils[coil_index]
    if not (0 <= point["r"] < current_coil.outer_radius - current_coil.inner_radius
            and 0 <= point["z"] < current_coil.height):
        raise ValueError(f"Point outside the coil mesh for Termopar {number}")
    analyzer = Analyzer(
        results=simulator.results_by_coil,
        simulator=simulator, furnace=furnace, coil_index=coil_index,
    )
    simulated = analyzer.get_point_data(point["r"], point["z"])
    experimental = np.genfromtxt(path, delimiter=",", skip_header=1, ndmin=2)
    experimental = experimental[np.isfinite(experimental[:, :2]).all(axis=1), :2]
    experimental = experimental[np.argsort(experimental[:, 0], kind="stable")]
    if not len(experimental):
        raise ValueError(f"No finite measurements in {path}")
    comparisons.append({
        "name": f"Termopar {number}",
        "point": point,
        "color": qualitative.Plotly[index % len(qualitative.Plotly)],
        "simulation": simulated,
        "experimental_hours": experimental[:, 0],
        "experimental_kelvin": experimental[:, 1] + 273.15,
    })

plot_height = 700

layout = {
    "title": {
        "text": "Comparación de temperaturas — simulación y termopares",
        "x": 0.5,
        "xanchor": "center",
        "y": 0.9,
        "yanchor": "top",
    },
    #"width": 1920,
    "height": plot_height,
    "font": {
        "family": "Arial",
        "size": 15,
        "color": "black",
    },
    "margin": {"t": int(plot_height * 0.25), "b": 100, "l": 100, "r": 100},
    "legend": {"x": 1.05, "xanchor": "left"},
    "xaxis": {"title": {"text": "Tiempo (h)"},
               "ticklabelstandoff": 15,
               "zeroline": False},
    "yaxis": {"title": {"text": "Temperatura (K)"}, "ticklabelstandoff": 15},
    "template": "none"
}

fig = go.Figure(layout=layout)
for comparison in comparisons:
    name = comparison["name"]
    point = comparison["point"]
    fig.add_trace(go.Scatter(
        x=simulation_hours, y=comparison["simulation"], mode="lines",
        name=f"{name} (simulación)", legendgroup=name,
        line={"color": comparison["color"], "width": 2, "dash": "solid"},
        hovertemplate=(
            f"{name} — rollo {point['coil_num']}, r={point['r']} m, z={point['z']} m<br>"
            "Tiempo: %{x:.2f} h<br>T: %{y:.2f} K<extra></extra>"
        ),
    ))
    fig.add_trace(go.Scatter(
        x=comparison["experimental_hours"], y=comparison["experimental_kelvin"],
        mode="lines", name=f"{name} (experimental)", legendgroup=name,
        line={"color": comparison["color"], "width": 2, "dash": "dot"},
    ))
fig.add_trace(go.Scatter(
    x=simulation_hours,
    y=[furnace.temperature_at(time) for time in simulator.times],
    mode="lines", name="Gas de protección",
    line={"color": "#555555", "width": 2, "dash": "dash"},
));


### Plot.

Comparación de todos los termopares: experimento punteado y simulación continua, con colores compartidos.

In [5]:
fig.show()

### Settings — Diferencias

In [6]:
# Uses the same points, colors and data prepared in the first Settings section.
# Compare at measurement times, within the simulation range (no extrapolation).
difference_layout = deepcopy(layout)
difference_layout["title"]["text"] = "Diferencia simulación y experimento"
difference_layout["yaxis"] = {
    "title": {"text": "Temperatura"},
    "ticklabelstandoff": 15,
    "zeroline": True,
    "zerolinecolor": "#999999",
    "zerolinewidth": 1,
}

difference_fig = go.Figure(layout=difference_layout)
for comparison in comparisons:
    hours = comparison["experimental_hours"]
    overlap = (hours >= simulation_hours[0]) & (hours <= simulation_hours[-1])
    if not overlap.any():
        raise ValueError(f"No overlapping times for {comparison['name']}")
    difference = (
        np.interp(hours[overlap], simulation_hours, comparison["simulation"])
        - comparison["experimental_kelvin"][overlap]
    )
    difference_fig.add_trace(go.Scatter(
        x=hours[overlap], y=difference, mode="lines",
        name=comparison["name"], legendgroup=comparison["name"],
        line={"color": comparison["color"], "width": 2},
        hovertemplate="Tiempo: %{x:.2f} h<br>ΔT: %{y:.2f} K<extra>%{fullData.name}</extra>",
    ))


### Plot — Diferencias
Valores positivos: simulación más caliente que el experimento.

In [7]:
difference_fig.show()

## Heatmap.

Muestra la sección radial–vertical de cada rollo, con el rollo 1 abajo y el 4 arriba. 

El selector muestra distintos tiempos, empezando en 0.5 h. La escala de color es común a todos los rollos y tiempos, en °C. La comparación anterior conserva sus unidades originales (K).


### Settings — Heatmap

Configura el tiempo inicial, los tiempos del selector y el diseño del mapa de calor.

In [8]:
# Snapshots share the simulator's time grid; no extra simulation is needed.
heatmap_time_hr = 0.5
snapshot_times_hr = np.unique(np.concatenate((
    [0.0, heatmap_time_hr, simulator.times[-1] / 3600.0],
    np.arange(1.0, simulator.times[-1] / 3600.0, 1.0),
)))
snapshot_indices = np.unique([
    int(np.argmin(np.abs(simulator.times / 3600.0 - time_hr)))
    for time_hr in snapshot_times_hr
])
initial_index = int(np.argmin(np.abs(simulator.times / 3600.0 - heatmap_time_hr)))
initial_step = int(np.flatnonzero(snapshot_indices == initial_index)[0])

# Histories already store one section: (height, radius, time).
sections_c = [history - 273.15 for history in simulator.results_by_coil]
if not all(np.isfinite(section).all() for section in sections_c):
    raise ValueError("La simulación contiene temperaturas no finitas; revisar la estabilidad antes de graficar.")
color_min = min(float(section.min()) for section in sections_c)
color_max = max(float(section.max()) for section in sections_c)
if color_max == color_min:
    color_max = color_min + 1.0

# Cell edges ensure the heatmap covers the physical dimensions exactly.
coil_meshes = []
stack_height = 0.0
for current_coil in furnace.coils:
    _, height_count, radial_count = current_coil.points.shape
    radial_edges = np.linspace(current_coil.inner_radius, current_coil.outer_radius, radial_count + 1)
    height_edges = stack_height + np.linspace(0, current_coil.height, height_count + 1)
    coil_meshes.append((radial_edges, height_edges))
    stack_height += current_coil.height + furnace.coil_sep_height
stack_height -= furnace.coil_sep_height


def coil_heatmaps(time_index):
    return [
        go.Heatmap(
            x=radial_edges,
            y=height_edges,
            z=section[:, :, time_index],
            coloraxis="coloraxis",
            zsmooth=False,
            name=f"Rollo {current_coil.coil_num}",
            hovertemplate=(
                f"Rollo {current_coil.coil_num}<br>"
                "Radio: %{x:.3f} m<br>Altura: %{y:.3f} m<br>"
                "Temperatura: %{z:.1f} °C<extra></extra>"
            ),
        )
        for current_coil, section, (radial_edges, height_edges)
        in zip(furnace.coils, sections_c, coil_meshes)
    ]


def heatmap_title(time_index):
    return f"Mapa de calor — {simulator.times[time_index] / 3600.0:.2f} h"


heatmap_fig = go.Figure(data=coil_heatmaps(initial_index))
heatmap_fig.frames = [
    go.Frame(
        name=str(int(time_index)),
        data=coil_heatmaps(time_index),
        traces=list(range(len(furnace.coils))),
        layout={"title": {"text": heatmap_title(time_index)}},
    )
    for time_index in snapshot_indices
]
heatmap_fig.update_layout(
    title={"text": heatmap_title(initial_index), "x": 0.5, "xanchor": "center"},
    width=700,
    height=1000,
    template="none",
    font={"family": "Arial", "size": 15, "color": "black"},
    margin={"t": 100, "b": 130, "l": 100, "r": 140},
    xaxis={"title": {"text": "Radio (m)"}, "zeroline": False},
    yaxis={"title": {"text": "Altura (m)"}, "range": [0, stack_height], "zeroline": False},
    coloraxis={
        "colorscale": "Jet", "cmin": color_min, "cmax": color_max,
        "colorbar": {"title": {"text": "T (°C)"}, "x": 1.25},
    },
    annotations=[
        {
            "x": 1.02, "xref": "paper", "y": (height_edges[0] + height_edges[-1]) / 2,
            "yref": "y", "text": f"Rollo {current_coil.coil_num}",
            "showarrow": False, "xanchor": "left",
        }
        for current_coil, (_, height_edges) in zip(furnace.coils, coil_meshes)
    ],
    sliders=[{
        "active": initial_step,
        "currentvalue": {"prefix": "Tiempo (h): "},
        "pad": {"t": 50},
        "steps": [
            {
                "label": f"{simulator.times[time_index] / 3600.0:.2f}",
                "method": "animate",
                "args": [[str(int(time_index))], {
                    "mode": "immediate", "frame": {"duration": 0, "redraw": True},
                    "transition": {"duration": 0},
                }],
            }
            for time_index in snapshot_indices
        ],
    }],
)

### Plot — Heatmap

In [9]:
heatmap_fig.show()